In [1]:
# import json
from pathlib import Path
import numpy as np
# import pytest
# import advanced_link_skdsp_v3_tx_flexible as txmod
import generate_tx_iq_dataset as genmod
import load_tx_iq_data as loadmod
from tx_controller_tone_pulse_stft_varlen import TonePulseTXControlNetVarLen, build_controlled_tone_pulse_from_variable_inputs
import torch
import math
import score_iq_decode as scorer
import argparse
import json
import tempfile
from pathlib import Path
from typing import Optional
import advanced_link_skdsp_v4_robust as link
import os

def repeat_to_length_mod(arr, target_length):
    if arr.ndim != 1:
        raise ValueError("Input array must be 1D")
    if len(arr) == 0:
        raise ValueError("Input array must not be empty")

    idx = np.arange(target_length) % len(arr)
    return arr[idx]


def delete_if_exists(filepath):
    """
    Check if a file exists and delete it.

    Parameters:
        filepath (str): Path to the file
    """
    if os.path.isfile(filepath):
        os.remove(filepath)
        print(f"Deleted: {filepath}")
    else:
        print(f"File not found: {filepath}")

# tmp_path = Path("local_test")
# out_root = tmp_path / "dataset"

# produced = genmod.generate_dataset(
#                                     output_root=out_root,
#                                     num_outputs=4,
#                                     min_total_samples=8192,
#                                     max_total_samples=12000,
#                                     section_len=1024,
#                                     num_sections=3,
#                                     seed=3,
#                                     random_payload_probability=0.5,
#                                     )

# assert len(produced) == 4

# sample_dirs = loadmod.list_sample_dirs(out_root)
# assert len(sample_dirs) == 4

# for sdir in sample_dirs:
#     assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
#     assert (sdir / "sections.npy").exists()
#     assert (sdir / "sections_meta.json").exists()

# whole = loadmod.load_whole_iq(sdir)
# sections = loadmod.load_sections(sdir)
# bundle = loadmod.load_sample_bundle(sdir)

# iq = whole["iq"]
# whole_meta = whole["meta"]
# secs = sections["sections"]
# sections_meta = sections["meta"]

# assert iq.dtype == np.complex64
# assert secs.dtype == np.complex64
# assert secs.shape == (3, 1024)

# assert whole_meta["actual_num_samples"] == len(iq)
# assert sections_meta["whole_num_samples"] == len(iq)
# assert len(sections_meta["starts"]) == 3

# for idx, start in enumerate(sections_meta["starts"]):
#     np.testing.assert_array_equal(secs[idx], iq[start:start + 1024])

# assert bundle["whole_iq"].shape == iq.shape
# assert bundle["sections"].shape == secs.shape



In [3]:
train_path = Path("train_set_00")
# test_path = Path("test_set_00")
# val_path = Path("val_set_00")

# for _ in [train_path, test_path, val_path]:
out_root = train_path / "dataset"
produced = genmod.generate_dataset(
                                    output_root=out_root,
                                    num_outputs=100,
                                    min_total_samples=20000,
                                    max_total_samples=100000,
                                    section_len=1024,
                                    num_sections=3,
                                    seed=3,
                                    random_payload_probability=0.0,
                                    )

sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
s

In [2]:
# "C:\Users\theon\Jamming\train_set_00"
train_path_dat = Path("C:/Users/theon/Jamming/train_set_00/dataset")
sample_dirs = loadmod.list_sample_dirs(train_path_dat)
sdir = sample_dirs[0]
whole = loadmod.load_whole_iq(sdir)
# for sdir in sample_dirs:
#     assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
#     assert (sdir / "sections.npy").exists()
#     assert (sdir / "sections_meta.json").exists()

# whole = loadmod.load_whole_iq(sdir)
# sections = loadmod.load_sections(sdir)
sdir

WindowsPath('C:/Users/theon/Jamming/train_set_00/dataset/sample_000000')

In [3]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

1.0

In [10]:
whole["meta"]

{'payload_source': 'message',
 'message_length': 881,
 'message': 'p 4oFZgwZl8 gxGIEK8VO5d28Y5  MjVq0yv8IH4 U 2adiyVWtcFuypDKxc  Fog FEuwO GblpNE1 LrH 7PY4oC2dwjq9Ap n48lV f3jw1ew0Twk1ZkTSBQQ U2lax2u44oYsTj6U5MdMJKI3Rk5y5bm1SD GaIB0dO6HmCGKd pStVdKxxydWfkQofePI95lAI85xM9 AsY6 kI867nr06V6VUIDUzBA8aT xMgEebl1S TEMK3Wsv JbD3Kd87NiY3vg6t9 RhyPpUx2FcGzEH0UoaEcYFuf Bu3E3OjaoCJFi qh2 zDT1haiEej tJp1 dTk8l nOaPoSqALMA1QRP 8Fpt0kBVexAfEIpqwT1pW5XPLj7GTbt3i npGXQ BjgcwdLtk2iN 04 Of5MvdemK1wxURLuQ2Tp  sRaS BL1D6vmK 7qpXgs0p6jakPyhMSW21mofyZKr3LsiFQwhdybuGYDPRpTJxd9ZW  iNZPjq8hLcZNVSJ1j9HJFaN OqMzrW hGb7QWJVZ115R0CmXx4 DrBt pl Wa J D8s2JOuGscJHoDPKU8EgyXJOor9 jWY8EszTCVQsfNJxugbJPRCCjOOW2LqmUecNMb4YLUC0EZDCG 4NXkBQb9SSFDKhUi8RzUqmC6ENiXV76DScKw kDRyEaWzmc HBk6ebqHNuWTlOMAzy820ZDis4dYt9TrmC0pyOfQuDbkH PX7 29MTEym3r77I1kMInyIv9ROaCtlXothVbG C fNMxz0e9SE0oWjNHCQRcsmSk74wP9LrpwboPQxO01FhQ8teYta4TZdF46FkjbyvQNDJUV2wHCc2djcALCCTl chbOrRtDVqtrJGKbyIxQLa',
 'random_bits': None,
 'random_seed': None,
 'pay

In [6]:
sdir

WindowsPath('C:/Users/theon/Jamming/train_set_00/dataset/sample_000004')

In [12]:
iq_path = Path(sdir / "whole_iq.npy")
meta_path = Path(sdir / "whole_meta.json")
print(f'sdir : {sdir}')
# assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
               
score = scorer.main(["--iq", str(iq_path), "--metadata", str(meta_path)])
score

sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000004
1.0


1.0

In [3]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
# whole_iq
rx_result_check = link.rx_command_iq(whole_iq, whole["meta"])
# score_check = scorer.score_decode(rx_result_check, whole["meta"])

fec : conv
baseband_offset_hz : 0.0


In [5]:
sections = loadmod.load_sections(sdir)
sections['sections'][0].shape

(1024,)

In [6]:
whole['iq'].shape[0]

123040

In [4]:
sdir_01 = sample_dirs[1]
whole_01 = loadmod.load_whole_iq(sdir_01)
whole_01['meta']['fec']


'conv'

In [3]:
sdir_01 = sample_dirs[4]
whole_01 = loadmod.load_whole_iq(sdir_01)
whole_01

{'iq': array([-0.01471287-0.0084325j , -0.00874602+0.00398263j,
        -0.00469382+0.00684296j, ...,  0.32975093-0.14786556j,
         0.32221863-0.15384778j, -0.01371755-0.01209665j],
       shape=(201792,), dtype=complex64),
 'meta': {'payload_source': 'message',
  'message_length': 881,
  'message': 'p 4oFZgwZl8 gxGIEK8VO5d28Y5  MjVq0yv8IH4 U 2adiyVWtcFuypDKxc  Fog FEuwO GblpNE1 LrH 7PY4oC2dwjq9Ap n48lV f3jw1ew0Twk1ZkTSBQQ U2lax2u44oYsTj6U5MdMJKI3Rk5y5bm1SD GaIB0dO6HmCGKd pStVdKxxydWfkQofePI95lAI85xM9 AsY6 kI867nr06V6VUIDUzBA8aT xMgEebl1S TEMK3Wsv JbD3Kd87NiY3vg6t9 RhyPpUx2FcGzEH0UoaEcYFuf Bu3E3OjaoCJFi qh2 zDT1haiEej tJp1 dTk8l nOaPoSqALMA1QRP 8Fpt0kBVexAfEIpqwT1pW5XPLj7GTbt3i npGXQ BjgcwdLtk2iN 04 Of5MvdemK1wxURLuQ2Tp  sRaS BL1D6vmK 7qpXgs0p6jakPyhMSW21mofyZKr3LsiFQwhdybuGYDPRpTJxd9ZW  iNZPjq8hLcZNVSJ1j9HJFaN OqMzrW hGb7QWJVZ115R0CmXx4 DrBt pl Wa J D8s2JOuGscJHoDPKU8EgyXJOor9 jWY8EszTCVQsfNJxugbJPRCCjOOW2LqmUecNMb4YLUC0EZDCG 4NXkBQb9SSFDKhUi8RzUqmC6ENiXV76DScKw kDRyEaWzmc HBk6ebq

In [4]:
sdir_01 = sample_dirs[4]
whole_01 = loadmod.load_whole_iq(sdir_01)
whole_iq = whole_01['iq']
# whole_iq
rx_result_check = link.rx_command_iq(whole_iq, whole_01["meta"])
rx_result_check

{'payload_bytes': b'p 4oFZgwZl8 gxGIEK8VO5d28Y5  MjVq0yv8IH4 U 2adiyVWtcFuypDKxc  Fog FEuwO GblpNE1 LrH 7PY4oC2dwjq9Ap n48lV f3jw1ew0Twk1ZkTSBQQ U2lax2u44oYsTj6U5MdMJKI3Rk5y5bm1SD GaIB0dO6HmCGKd pStVdKxxydWfkQofePI95lAI85xM9 AsY6 kI867nr06V6VUIDUzBA8aT xMgEebl1S TEMK3Wsv JbD3Kd87NiY3vg6t9 RhyPpUx2FcGzEH0UoaEcYFuf Bu3E3OjaoCJFi qh2 zDT1haiEej tJp1 dTk8l nOaPoSqALMA1QRP 8Fpt0kBVexAfEIpqwT1pW5XPLj7GTbt3i npGXQ BjgcwdLtk2iN 04 Of5MvdemK1wxURLuQ2Tp  sRaS BL1D6vmK 7qpXgs0p6jakPyhMSW21mofyZKr3LsiFQwhdybuGYDPRpTJxd9ZW  iNZPjq8hLcZNVSJ1j9HJFaN OqMzrW hGb7QWJVZ115R0CmXx4 DrBt pl Wa J D8s2JOuGscJHoDPKU8EgyXJOor9 jWY8EszTCVQsfNJxugbJPRCCjOOW2LqmUecNMb4YLUC0EZDCG 4NXkBQb9SSFDKhUi8RzUqmC6ENiXV76DScKw kDRyEaWzmc HBk6ebqHNuWTlOMAzy820ZDis4dYt9TrmC0pyOfQuDbkH PX7 29MTEym3r77I1kMInyIv9ROaCtlXothVbG C fNMxz0e9SE0oWjNHCQRcsmSk74wP9LrpwboPQxO01FhQ8teYta4TZdF46FkjbyvQNDJUV2wHCc2djcALCCTl chbOrRtDVqtrJGKbyIxQLa',
 'payload_len': 881,
 'message': 'p 4oFZgwZl8 gxGIEK8VO5d28Y5  MjVq0yv8IH4 U 2adiyVWtcFuypDKxc  

In [6]:
score_check = scorer.score_decode(rx_result_check, whole_01["meta"])
score_check

1.0

In [5]:
iq_path = Path(sample_dirs[4] / "whole_iq.npy")
meta_path = Path(sample_dirs[4] / "whole_meta.json")
print(f'sdir : {sample_dirs[4]}')
# assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
               
score = scorer.main(["--iq", str(iq_path), "--metadata", str(meta_path)])
score

sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000004
1.0


1.0

In [ ]:
1000000000
1000000000

In [24]:
whole['meta']['sample_rate_hz']

1000000000.0

In [8]:
device = "cuda"

epochs = 2
jammer_sampling_freq = 2e9
scoring_dict = {}

model = TonePulseTXControlNetVarLen(in_ch=4, base_ch=16, n_scalar_features=6, max_tones=8).to(device)
   
for epoch in range(epochs):
    epoch_score = 0.0
    print(f'epoch : {epoch}')
    sample_dirs = loadmod.list_sample_dirs(train_path_dat)
    for sdir in sample_dirs:
        whole = loadmod.load_whole_iq(sdir)
        
        whole_iq = whole['iq']
        sections = loadmod.load_sections(sdir)
        iq1 = sections['sections'][0]
        iq2 = sections['sections'][1]
        iq3 = sections['sections'][2]
        whole_sample_rate = whole['meta']['sample_rate_hz']
        
        # rx_result_check = link.rx_command_iq(whole_iq, whole["meta"])
        # score_check = scorer.score_decode(rx_result_check, whole["meta"])
        
        # print('resampling 1')
        iq1 = link.resample_iq(iq1, whole_sample_rate, jammer_sampling_freq)[:1024]
        # print('resampling 2')
        iq2 = link.resample_iq(iq2, whole_sample_rate, jammer_sampling_freq)[:1024]
        # print('resampling 3')
        iq3 = link.resample_iq(iq3, whole_sample_rate, jammer_sampling_freq)[:1024]
        
        # print('jam_iq')
        jam_iq =  build_controlled_tone_pulse_from_variable_inputs(
                                                                    model=model,
                                                                    rx_iq_windows=[iq1, iq2, iq3],
                                                                    intake_sample_rate_hz=jammer_sampling_freq,
                                                                    desired_output_iq_len=100_000,
                                                                    user_peak_power_fraction=0.1,
                                                                    seed=11,
                                                                    device=device,
                                                                )

        # print('jam resampling')

        jam_iq_rx_resam = link.resample_iq(jam_iq['tx_iq'], jammer_sampling_freq, whole_sample_rate)
        jam_iq_rx_resam = repeat_to_length_mod(jam_iq_rx_resam, whole_iq.shape[0])
        jammed = whole_iq + jam_iq_rx_resam[:whole_iq.shape[0]]
        jam_path = sdir / "jammed_iq.npy"
        meta_path = sdir / "whole_meta.json"
        score = 0.0
        try:
            rx_result = link.rx_command_iq(jammed, whole["meta"])
            score = scorer.score_decode(rx_result, whole["meta"])
        except RuntimeError as e:
            estr = e.__str__() 
            if estr == "No valid packet found after acquisition, header decode, FEC decode, and CRC":
                print(estr)
                rx_result_check = link.rx_command_iq(whole_iq, whole["meta"])
                score_check = scorer.score_decode(rx_result_check, whole["meta"])
                print(f"score_check : {score_check}")
                pass
            else:
                raise e
        epoch_score += score
        print(f'sdir : {sdir} score: {score}')


    scoring_dict[epoch] = epoch_score

epoch : 0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000000 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000001 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000002 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000003 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000004 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000005 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000006 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000007 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000008 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000009 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000010 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000011 score: 1.0
sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000012 score: 1.0
sd

In [9]:
scoring_dict

{0: 100.0}

In [22]:
device = "cuda"

epochs = 3
jammer_sampling_freq = 2e9
scoring_dict = {}

model = TonePulseTXControlNetVarLen(in_ch=4, base_ch=16, n_scalar_features=6, max_tones=8).to(device)
returns = []
criterion = JammerLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
loop_limit=20

for epoch in range(epochs):
    # epoch_score = 0.0
    print(f'epoch : {epoch}')
    sample_dirs = loadmod.list_sample_dirs(train_path_dat)
    scores=[]
    loop_limiter = 0 
    for sdir in sample_dirs:
        if loop_limiter == loop_limit:
            break
        whole = loadmod.load_whole_iq(sdir)
        loop_limiter+=1
        whole_iq = whole['iq']
        sections = loadmod.load_sections(sdir)
        iq1 = sections['sections'][0]
        iq2 = sections['sections'][1]
        iq3 = sections['sections'][2]
        whole_sample_rate = whole['meta']['sample_rate_hz']
        
        # rx_result_check = link.rx_command_iq(whole_iq, whole["meta"])
        # score_check = scorer.score_decode(rx_result_check, whole["meta"])
        
        # print('resampling 1')
        iq1 = link.resample_iq(iq1, whole_sample_rate, jammer_sampling_freq)[:1024]
        # print('resampling 2')
        iq2 = link.resample_iq(iq2, whole_sample_rate, jammer_sampling_freq)[:1024]
        # print('resampling 3')
        iq3 = link.resample_iq(iq3, whole_sample_rate, jammer_sampling_freq)[:1024]
        
        # print('jam_iq')
        jam_iq =  build_controlled_tone_pulse_from_variable_inputs(
                                                                    model=model,
                                                                    rx_iq_windows=[iq1, iq2, iq3],
                                                                    intake_sample_rate_hz=jammer_sampling_freq,
                                                                    desired_output_iq_len=100_000,
                                                                    user_peak_power_fraction=6.0,
                                                                    seed=11,
                                                                    device=device,
                                                                )

        # print('jam resampling')

        jam_iq_rx_resam = link.resample_iq(jam_iq['tx_iq'], jammer_sampling_freq, whole_sample_rate)
        jam_iq_rx_resam = repeat_to_length_mod(jam_iq_rx_resam, whole_iq.shape[0])
        jammed = whole_iq + jam_iq_rx_resam[:whole_iq.shape[0]]
        scores.append(criterion(torch.Tensor(jammed), whole))
       
        # epoch_score += scores[-1].item()
        print(f'epoch : {epoch}  sdir : {sdir} score: {scores[-1].item()}')

    loss = torch.stack(scores).sum()/len(scores)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    scoring_dict[epoch] = loss
    print(scoring_dict)

scoring_dict

epoch : 0


C:\Users\theon\AppData\Local\Temp\ipykernel_24348\3869851707.py:60: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Copy.cpp:308.)
  scores.append(criterion(torch.Tensor(jammed), whole))


AttributeError: 'Tensor' object has no attribute 'astype'

In [19]:
type(whole_iq)

numpy.ndarray

In [18]:
scores[0].shape

torch.Size([1])

In [20]:
import torch
import torch.nn as nn


class JammerLossFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, jammed, whole):
        ctx.whole = whole
        ctx.save_for_backward(jammed.detach())

        score = JammerLossFunction._evaluate(jammed, whole)
        return jammed.new_tensor(score)

    @staticmethod
    def backward(ctx, grad_output):
        jammed, = ctx.saved_tensors
        whole = ctx.whole

        eps = 1e-4
        grad_jammed = torch.zeros_like(jammed)

        flat = jammed.reshape(-1)
        flat_grad = grad_jammed.reshape(-1)

        for i in range(flat.numel()):
            x_plus = jammed.clone()
            x_minus = jammed.clone()

            x_plus.reshape(-1)[i] += eps
            x_minus.reshape(-1)[i] -= eps

            f_plus = JammerLossFunction._evaluate(x_plus, whole)
            f_minus = JammerLossFunction._evaluate(x_minus, whole)

            flat_grad[i] = (f_plus - f_minus) / (2 * eps)

        grad_jammed = grad_jammed * grad_output
        return grad_jammed, None

    @staticmethod
    def _evaluate(jammed, whole):
        score = 0.0
        try:
            rx_result = link.rx_command_iq(jammed, whole["meta"])
            score = scorer.score_decode(rx_result, whole["meta"])
        except RuntimeError as e:
            estr = str(e)
            if estr == "No valid packet found after acquisition, header decode, FEC decode, and CRC":
                score = 0.0
            else:
                raise
        return float(score)


class JammerLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, jammed, whole):
        return JammerLossFunction.apply(jammed, whole)


# Example usage
# pred = torch.tensor([2.5, 0.0, 2.1], requires_grad=True)
# target = torch.tensor([3.0, -0.5, 2.0])

# criterion = CustomLoss(alpha=0.05)
# loss = criterion(pred, target)

# print("Loss:", loss.item())

# # Backprop
# loss.backward()
# print("Gradients:", pred.grad)


In [ ]:
def compute_returns(rewards, gamma: float):
    """
    Compute discounted returns for one episode.
    """
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns


def train_policy_gradient(
    env,
    policy: nn.Module,
    optimizer: optim.Optimizer,
    num_episodes: int = 1000,
    gamma: float = 0.99,
    device: str = "cpu",
    max_steps_per_episode: int = 1000,
    normalize_returns: bool = True,
    print_every: int = 50,
):
    """
    Generic REINFORCE training loop.

    Args:
        env: Gym-like environment with reset() and step(action)
        policy: PyTorch model mapping observation -> action logits
        optimizer: PyTorch optimizer
        num_episodes: number of training episodes
        gamma: discount factor
        device: torch device
        max_steps_per_episode: safety cap per episode
        normalize_returns: whether to normalize discounted returns
        print_every: logging frequency
    """
    device = torch.device(device)
    policy.to(device)

    reward_history = []

    for episode in range(1, num_episodes + 1):
        # Gymnasium reset API compatibility
        reset_result = env.reset()
        state = reset_result[0] if isinstance(reset_result, tuple) else reset_result

        log_probs = []
        rewards = []
        total_reward = 0.0

        for step in range(max_steps_per_episode):
            action, log_prob = select_action(policy, state, device)

            step_result = env.step(action)
            if len(step_result) == 5:
                next_state, reward, terminated, truncated, _ = step_result
                done = terminated or truncated
            else:
                next_state, reward, done, _ = step_result

            log_probs.append(log_prob)
            rewards.append(reward)
            total_reward += reward
            state = next_state

            if done:
                break

        # Compute discounted returns
        returns = compute_returns(rewards, gamma)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        if normalize_returns and len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Policy gradient loss
        loss = []
        for log_prob, G in zip(log_probs, returns):
            loss.append(-log_prob * G)
        loss = torch.stack(loss).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reward_history.append(total_reward)

        if episode % print_every == 0:
            avg_reward = sum(reward_history[-print_every:]) / print_every
            print(
                f"Episode {episode:4d} | "
                f"Avg Reward: {avg_reward:8.3f} | "
                f"Last Reward: {total_reward:8.3f} | "
                f"Loss: {loss.item():8.3f}"
            )

    return reward_history